# ML-04b : validité statistique des comparaisons de modèles (C#)

**Navigation** : [Index](README.md) | [ML-04](ML-04-Evaluation-CSharp.ipynb) | [Suivant ML-05](ML-05-TimeSeries-CSharp.ipynb) | [Version Python](ML-04b-ModelComparison-Validity-Python.ipynb)

## Objectifs

Comparer des scores sur plusieurs splits, mesurer un intervalle bootstrap apparié, examiner un test corrigé et ajuster le seuil pour trois comparaisons.

**Prérequis** : C#, régression et probabilités ; CPython avec NumPy et scikit-learn : le chargeur partagé `Config/PythonNetLoader.cs` retient le premier interpréteur qui importe scikit-learn, et `PYTHONNET_PYDLL` reste prioritaire s'il est défini. **Durée indicative** : 90 minutes.

Les mêmes observations diabetes et indices de split que Python sont chargés via Python.NET. Les entraînements sont natifs : moindres carrés sans pénalisation par QR MathNet, forêt et arbres boostés ML.NET. Les hyperparamètres et algorithmes des arbres diffèrent de sklearn, donc leurs scores ne sont pas attendus identiques.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
#r "nuget: Microsoft.ML, 4.0.2"
#r "nuget: Microsoft.ML.FastTree, 4.0.2"
#r "nuget: PythonNet, 3.0.5"
#load "../../Config/PythonNetLoader.cs"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text.Json;
using System.Globalization;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Trainers.FastTree;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.Distributions;
using Python.Runtime;
// Culture invariante pour TOUS les threads du noyau : chaque cellule peut s'executer sur un
// thread different, et CurrentCulture seul ne vaut que pour le thread de cette cellule
// (sous une machine fr-FR, les cellules suivantes afficheraient 0,4849 au lieu de 0.4849).
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
// CPython de pythonnet : PYTHONNET_PYDLL s'il est defini, sinon le premier interpreteur qui importe sklearn (voir PythonNetLoader.cs)
if (PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "sklearn") != null) PythonEngine.BeginAllowThreads();
Console.WriteLine($".NET {Environment.Version} | ML.NET {typeof(MLContext).Assembly.GetName().Version} | MathNet {typeof(Matrix<double>).Assembly.GetName().Version} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");

Installed Packages MathNet.Numerics, 5.0.0 Microsoft.ML, 4.0.2 Microsoft.ML.FastTree, 4.0.2 PythonNet, 3.0.5

.NET 10.0.14 | ML.NET 1.0.0.0 | MathNet 5.0.0.0 | Python.NET 3.0.5.0


Le pont charge uniquement les données, les indices sklearn et une prédiction de référence de régression linéaire ; les trois modèles du carnet sont ajustés côté C#. Le témoin permet de contrôler le QR sur le split initial.

In [2]:
string payload;
using (Py.GIL()) {
    using var scope = Py.CreateScope();
    scope.Exec(@"import json, numpy as np, sklearn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
D=load_diabetes(); X=D.data; y=D.target; ids=np.arange(len(y))
tr,te=train_test_split(ids,test_size=.25,random_state=42)
splits=[train_test_split(ids,test_size=.25,random_state=s) for s in range(25)]
folds=list(KFold(n_splits=10,shuffle=True,random_state=42).split(X))
ref=LinearRegression().fit(X[tr],y[tr]).predict(X[te])
payload=json.dumps(dict(X=X.tolist(),y=y.tolist(),tr=tr.tolist(),te=te.tolist(),splits=[[a.tolist(),b.tolist()] for a,b in splits],folds=[[a.tolist(),b.tolist()] for a,b in folds],ref=ref.tolist(),version=sklearn.__version__))");
    payload=scope.Get("payload").As<string>();
}
var doc=JsonDocument.Parse(payload); var root=doc.RootElement;
double[][] X=root.GetProperty("X").EnumerateArray().Select(r=>r.EnumerateArray().Select(v=>v.GetDouble()).ToArray()).ToArray();
double[] y=root.GetProperty("y").EnumerateArray().Select(v=>v.GetDouble()).ToArray();
int[] Ind(JsonElement e)=>e.EnumerateArray().Select(v=>v.GetInt32()).ToArray();
(int[] train,int[] test) Pair(JsonElement e)=>(Ind(e[0]),Ind(e[1]));
var first=(train:Ind(root.GetProperty("tr")),test:Ind(root.GetProperty("te")));
var splits=root.GetProperty("splits").EnumerateArray().Select(Pair).ToArray();
var folds=root.GetProperty("folds").EnumerateArray().Select(Pair).ToArray();
var reference=root.GetProperty("ref").EnumerateArray().Select(v=>v.GetDouble()).ToArray();
Console.WriteLine($"Diabetes {X.Length} × {X[0].Length}, sklearn {root.GetProperty("version").GetString()}, test {first.test.Length}, splits {splits.Length}, plis {folds.Length}");

Diabetes 442 × 10, sklearn 1.9.1, test 111, splits 25, plis 10


## 1. Un split puis 25 splits

Les mêmes indices sont utilisés pour les trois modèles. Les arbres ML.NET sont de vraies implémentations de forêt et de boosting ; 100 arbres sont entraînés dans chaque cas. Les réglages et le générateur aléatoire diffèrent de sklearn.

La forêt autorise jusqu’à 442 feuilles, avec au moins une observation par feuille et toutes les variables. FastTree utilise huit feuilles au plus et un taux de 0,1 ; huit feuilles ne reproduisent pas exactement une profondeur maximale de trois. Les règles de découpage et d’échantillonnage des moteurs restent différentes.


In [3]:
public class DataRow { [VectorType(10)] public float[] Features {get;set;} = Array.Empty<float>(); public float Label {get;set;} }
public class ScoredRow { public float Score {get;set;} }
var ml = new MLContext(seed:42);
string[] names={"LinearRegression","RandomForest","GradientBoosting"};
double[] FitPredict(string name,int[] train,int[] test) {
    if(name==names[0]) {
        var a=Matrix<double>.Build.Dense(train.Length,11,(i,j)=>j==0?1:X[train[i]][j-1]);
        var b=Vector<double>.Build.DenseOfEnumerable(train.Select(i=>y[i]));
        var beta=a.QR().Solve(b);
        return test.Select(i=>beta[0]+Enumerable.Range(0,10).Sum(j=>beta[j+1]*X[i][j])).ToArray();
    }
    var rows=train.Select(i=>new DataRow{Features=X[i].Select(v=>(float)v).ToArray(),Label=(float)y[i]}).ToArray();
    var data=ml.Data.LoadFromEnumerable(rows);
    IEstimator<ITransformer> trainer=name==names[1]
      ? ml.Regression.Trainers.FastForest(new FastForestRegressionTrainer.Options{NumberOfTrees=100,NumberOfLeaves=442,MinimumExampleCountPerLeaf=1,FeatureFraction=1,FeatureFractionPerSplit=1,NumberOfThreads=1})
      : ml.Regression.Trainers.FastTree(new FastTreeRegressionTrainer.Options{NumberOfTrees=100,NumberOfLeaves=8,MinimumExampleCountPerLeaf=1,LearningRate=0.1,NumberOfThreads=1});
    var model=trainer.Fit(data);
    var testData=ml.Data.LoadFromEnumerable(test.Select(i=>new DataRow{Features=X[i].Select(v=>(float)v).ToArray(),Label=(float)y[i]}));
    return ml.Data.CreateEnumerable<ScoredRow>(model.Transform(testData),false).Select(p=>(double)p.Score).ToArray();
}
double R2(double[] actual,double[] pred) {
 if(actual.Length<2 || actual.Length!=pred.Length) throw new ArgumentException("R² : longueurs égales d'au moins deux observations");
 double m=actual.Average(), ss=actual.Sum(a=>(a-m)*(a-m));
 double err=actual.Zip(pred,(a,p)=>(a-p)*(a-p)).Sum();
 // Convention sklearn force_finite=true pour une cible constante.
 return ss==0 ? (err==0 ? 1 : 0) : 1-err/ss;
}
if(R2(new[]{2.0,2.0},new[]{2.0,2.0})!=1 || R2(new[]{2.0,2.0},new[]{1.0,2.0})!=0) throw new Exception("R² cible constante");
double[] Truth(int[] idx)=>idx.Select(i=>y[i]).ToArray();
var predictions=names.ToDictionary(n=>n,n=>FitPredict(n,first.train,first.test));
double maxOls=predictions[names[0]].Zip(reference,(a,b)=>Math.Abs(a-b)).Max();
if(maxOls>1e-6) throw new Exception($"QR/sklearn discordance {maxOls}");
Console.WriteLine($"QR vs sklearn écart maximal {maxOls:E2}");
foreach(var n in names) Console.WriteLine($"{n,-18} R² = {R2(Truth(first.test),predictions[n]):F4}");

QR vs sklearn écart maximal 3.41E-013


LinearRegression   R² = 0.4849


RandomForest       R² = 0.4448


GradientBoosting   R² = 0.4217


L'écart QR / sklearn contrôle la régression linéaire sur ce split. Un classement sur une seule partition ne prouve pas une supériorité générale ; répétons exactement les 25 graines du carnet source.

In [4]:
var tally=names.ToDictionary(n=>n,n=>0);
foreach(var (train,test) in splits) {
 var score=names.ToDictionary(n=>n,n=>R2(Truth(test),FitPredict(n,train,test)));
 tally[score.OrderByDescending(kv=>kv.Value).First().Key]++;
}
foreach(var n in names) Console.WriteLine($"{n,-18} : {tally[n]} / {splits.Length} splits");

LinearRegression   : 25 / 25 splits


RandomForest       : 0 / 25 splits


GradientBoosting   : 0 / 25 splits


La régression linéaire gagne les 25 partitions de cette exécution. Les gagnants résument ces partitions qui se recouvrent. Ils ne constituent pas 25 expériences indépendantes. L'ordre peut différer de Python car les arbres et leurs réglages diffèrent.

### Exercice 1 — Quatre modèles

Ajoutez un quatrième modèle sur les mêmes plis. Pour six p-values fournies, la fonction doit retourner une décision par paire avec le seuil Bonferroni ; les deux fixtures ont des résultats différents.

In [5]:
Dictionary<string,bool> BonferroniExercise(Dictionary<string,double> pvalues) {
    // TODO : verifier les p-values, puis retourner une décision pour chaque paire.
    return null;
}
foreach(var fixture in new[]{
    new Dictionary<string,double>{{"AB",.001},{"AC",.020},{"AD",.006},{"BC",.008},{"BD",.040},{"CD",.009}},
    new Dictionary<string,double>{{"AB",.5},{"AC",.002},{"AD",.7},{"BC",.003},{"BD",.8},{"CD",.004}}}) {
 var answer=BonferroniExercise(fixture);
 if(answer==null) Console.WriteLine("Exercice 1 à compléter");
 else Console.WriteLine($"Contrôle : {answer.Count==fixture.Count && fixture.All(kv=>answer.ContainsKey(kv.Key) && answer[kv.Key]==(kv.Value<.05/fixture.Count))}");
}

Exercice 1 à compléter


Exercice 1 à compléter


## 2. Dix plis

La moyenne et l'écart-type décrivent les scores, mais leur chevauchement ne constitue pas un test. Les ensembles d'entraînement se recouvrent et les scores ne sont pas indépendants.

In [6]:
record ScorePlot(string Content);
var cv=names.ToDictionary(n=>n,n=>folds.Select(f=>R2(Truth(f.test),FitPredict(n,f.train,f.test))).ToArray());
double Mean(double[] a)=>a.Average();
double Var(double[] a) { double m=Mean(a); return a.Sum(v=>(v-m)*(v-m))/(a.Length-1); }
foreach(var n in names) Console.WriteLine($"{n,-18} R² = {Mean(cv[n]):F4} ± {Math.Sqrt(Var(cv[n]) * (cv[n].Length - 1.0) / cv[n].Length):F4} ; min {cv[n].Min():F3}, max {cv[n].Max():F3}");

// Moyennes et ecarts-types de chaque famille : figure statique sur fond opaque.
var plot=new System.Text.StringBuilder();
plot.Append("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='340' viewBox='0 0 680 340'><rect width='680' height='340' fill='white'/><text x='340' y='24' text-anchor='middle' font-family='sans-serif' font-size='16'>R² par modèle : moyenne ± écart-type</text><line x1='95' y1='260' x2='635' y2='260' stroke='black'/><line x1='95' y1='50' x2='95' y2='260' stroke='black'/>");
for(int j=0;j<=5;j++) { double v=j*.15; double py=260-v/0.75*200; plot.Append($"<line x1='90' y1='{py:F1}' x2='95' y2='{py:F1}' stroke='black'/><text x='84' y='{py+4:F1}' text-anchor='end' font-size='11'>{v:F2}</text>"); }
string[] colors={"#195f96","#b35616","#548139"};
for(int i=0;i<names.Length;i++) { double x=180+i*185, m=Mean(cv[names[i]]), sd=Math.Sqrt(Var(cv[names[i]]) * (cv[names[i]].Length - 1.0) / cv[names[i]].Length); double py=260-m/0.75*200, hi=260-(m+sd)/0.75*200, lo=260-(m-sd)/0.75*200; plot.Append($"<line x1='{x:F1}' y1='{hi:F1}' x2='{x:F1}' y2='{lo:F1}' stroke='{colors[i]}' stroke-width='3'/><line x1='{x-7:F1}' y1='{hi:F1}' x2='{x+7:F1}' y2='{hi:F1}' stroke='{colors[i]}'/><line x1='{x-7:F1}' y1='{lo:F1}' x2='{x+7:F1}' y2='{lo:F1}' stroke='{colors[i]}'/><circle cx='{x:F1}' cy='{py:F1}' r='6' fill='{colors[i]}'/><text x='{x:F1}' y='282' text-anchor='middle' font-family='sans-serif' font-size='12'>{names[i]}</text>"); }
plot.Append("<text x='365' y='320' text-anchor='middle' font-family='sans-serif' font-size='12'>Modèle (moyenne et dispersion sur 10 plis)</text><text transform='translate(25 155) rotate(-90)' text-anchor='middle' font-family='sans-serif' font-size='12'>R² (sans unité)</text></svg>");
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<ScorePlot>((v,w)=>w.Write(v.Content),"text/html");
display(new ScorePlot(plot.ToString()));

LinearRegression   R² = 0.4649 ± 0.1136 ; min 0.299, max 0.602


RandomForest       R² = 0.3620 ± 0.1382 ; min 0.061, max 0.538


GradientBoosting   R² = 0.3628 ± 0.1715 ; min -0.009, max 0.541


R² par modèle : moyenne ± écart-type 0.00 0.15 0.30 0.45 0.60 0.75 LinearRegression RandomForest GradientBoosting Modèle (moyenne et dispersion sur 10 plis) R² (sans unité)

Sur les dix plis, la régression linéaire obtient 0,4649 ± 0,1136, la forêt 0,3620 ± 0,1382 et le boosting 0,3628 ± 0,1715. Les écarts-types utilisent le diviseur 10, comme `numpy.std` par défaut. L'écart-type décrit la dispersion entre plis ; ce n'est pas un intervalle de confiance.

## 3. Bootstrap apparié

Sur le split initial, les prédictions des deux modèles sont déjà calculées. Pour une cible constante dans un rééchantillon, on suit la convention sklearn `force_finite=True` : R² vaut 1 si la prédiction est parfaite et 0 sinon. On rééchantillonne ensemble les observations de test 2000 fois : l'intervalle obtenu est conditionnel aux modèles ajustés et suppose des observations adaptées au bootstrap i.i.d.

In [7]:
double Quantile(double[] v,double q) { var a=v.OrderBy(x=>x).ToArray(); double p=(a.Length-1)*q; int i=(int)p; return a[i]+(a[Math.Min(i+1,a.Length-1)]-a[i])*(p-i); }
var rng=new Random(42); var ya=Truth(first.test); var pa=predictions[names[1]]; var pb=predictions[names[2]];
var diffs=new double[2000];
for(int b=0;b<diffs.Length;b++) { var ix=Enumerable.Range(0,ya.Length).Select(_=>rng.Next(ya.Length)).ToArray(); var yt=ix.Select(i=>ya[i]).ToArray(); diffs[b]=R2(yt,ix.Select(i=>pa[i]).ToArray())-R2(yt,ix.Select(i=>pb[i]).ToArray()); }
Console.WriteLine($"RF - GB moyen {diffs.Average():F4} ; IC percentile 95 % [{Quantile(diffs,.025):F4}, {Quantile(diffs,.975):F4}] ; B={diffs.Length}");

RF - GB moyen 0.0235 ; IC percentile 95 % [-0.0527, 0.1111] ; B=2000


L'intervalle observé pour forêt moins boosting est [-0,0527 ; 0,1111]. L'intervalle porte sur la différence des métriques calculées avec les mêmes indices, et ne comprend pas la variabilité du réentraînement.

### Exercice 2 — IC de RMSE seul et de différence appariée

Construisez les deux IC percentile : le premier pour le RMSE d'un modèle, le second pour la différence des deux RMSE. Le fournisseur reçoit le numéro de tirage et la graine ; utilisez les mêmes indices pour les deux modèles. Vérifiez dimensions, indices et valeurs finies avant de calculer. Les fixtures changent les données et la graine ; sur le test, utilisez 1000 tirages.

In [8]:
double[] BootstrapRmseExercise(double[] actual,double[] a,double[] b,int nBoot,int seed,Func<int,int,int[]> provider) {
    // TODO : pour chaque tirage fourni, calculer RMSE(a) et RMSE(a)-RMSE(b),
    // puis retourner les percentiles 2,5 et 97,5 pour chaque série.
    return null;
}
var samples=new[]{new[]{0,0,1},new[]{0,1,2},new[]{2,2,1},new[]{1,0,1}};
foreach(var f in new[]{(y:new[]{0.0,2.0,5.0},a:new[]{1.0,1.0,4.0},b:new[]{0.0,3.0,5.0},seed:0),
                        (y:new[]{1.0,3.0,8.0},a:new[]{1.0,5.0,7.0},b:new[]{3.0,3.0,8.0},seed:1)}) {
 int[] Provider(int iteration,int seed)=>samples[(iteration+seed)%samples.Length];
 var rA=new List<double>(); var rDiff=new List<double>();
 for(int i=0;i<samples.Length;i++) { var ix=Provider(i,f.seed);
  double ra=Math.Sqrt(ix.Average(j=>Math.Pow(f.y[j]-f.a[j],2)));
  double rb=Math.Sqrt(ix.Average(j=>Math.Pow(f.y[j]-f.b[j],2)));
  rA.Add(ra); rDiff.Add(ra-rb);
 }
 var expected=new[]{Quantile(rA.ToArray(),.025),Quantile(rA.ToArray(),.975),Quantile(rDiff.ToArray(),.025),Quantile(rDiff.ToArray(),.975)};
 var answer=BootstrapRmseExercise(f.y,f.a,f.b,samples.Length,f.seed,Provider);
 if(answer==null) Console.WriteLine("Exercice 2 à compléter");
 else Console.WriteLine($"Contrôle : {answer.Length==4 && answer.All(double.IsFinite) && answer.Zip(expected,(x,z)=>Math.Abs(x-z)<1e-10).All(x=>x)}");
}
// Sur le test, fournir 1000 tirages d'indices générés avec une graine fixée.

Exercice 2 à compléter


Exercice 2 à compléter


## 4. Test corrigé et comparaisons multiples

Pour les différences appariées $d$, la variance approchée de leur moyenne est $s_d^2(1/k+n_{test}/n_{train})$. Cette correction tient compte de la dépendance, sans garantir un niveau exact. Les tests sont unilatéraux dans l'ordre indiqué. Pour trois paires, le seuil Bonferroni est 0,05/3.

In [9]:
(double t,double p,double effect) Compare(double[] a,double[] b,double ratio=44.0/398) {
 if(a==null || b==null || a.Length!=b.Length || a.Length<2 || !a.All(double.IsFinite) || !b.All(double.IsFinite) || !double.IsFinite(ratio) || ratio<=0)
     throw new ArgumentException("scores finis de même taille >= 2 et ratio positif requis");
 var d=a.Zip(b,(u,v)=>u-v).ToArray(); double m=Mean(d), v=Var(d);
 double t=v==0 ? (m==0 ? 0 : Math.CopySign(double.PositiveInfinity,m)) : m/Math.Sqrt(v*(1.0/d.Length+ratio));
 double p=double.IsPositiveInfinity(t) ? 0 : double.IsNegativeInfinity(t) ? 1 : StudentT.CDF(0,1,d.Length-1,-t);
 double effect=(m==0 && v==0) ? 0 : m/Math.Sqrt(m*m+v); return (t,p,effect);
}
foreach(var (a,b) in new[]{(names[0],names[1]),(names[0],names[2]),(names[1],names[2])}) {
 var z=Compare(cv[a],cv[b]); Console.WriteLine($"{a} - {b}: t={z.t:F3}, p unilatérale={z.p:F4}, effet borné={z.effect:F3}, seuil={0.05/3:F4}");
}
var oracle=Compare(new[]{.4,.2,0.0,.2,.2},new[]{0.0,0.0,0.0,0.0,0.0},.25);
var inverse=Compare(new[]{0.0,0.0,0.0,0.0,0.0},new[]{.4,.2,0.0,.2,.2},.25);
if(Math.Abs(oracle.t-2.1081851067789197)>1e-10 || Math.Abs(oracle.p-.051350213747755905)>1e-10 || Math.Abs(inverse.p-.948649786252244)>1e-10 || Math.Abs(oracle.effect-.816496580927726)>1e-10) throw new Exception("Contrôle indépendant Nadeau–Bengio");
if(!double.IsFinite(StudentT.CDF(0,1,4,1e6)) || !double.IsFinite(StudentT.CDF(0,1,4,-1e6))) throw new Exception("CDF t non finie");
Console.WriteLine($"Contrôle indépendant : t={oracle.t:F6}, p={oracle.p:F6}, p inverse={inverse.p:F6}, effet={oracle.effect:F6}");
var zero=Compare(new[]{0.0,0.0},new[]{0.0,0.0});
var pos=Compare(new[]{1.0,1.0},new[]{0.0,0.0});
var neg=Compare(new[]{0.0,0.0},new[]{1.0,1.0});
if(zero.p!=.5 || zero.effect!=0 || pos.p!=0 || pos.effect!=1 || neg.p!=1 || neg.effect!=-1) throw new Exception("Conventions variance nulle");
bool rejected=false; try { Compare(new[]{1.0,2.0},new[]{1.0}); } catch(ArgumentException) { rejected=true; }
if(!rejected) throw new Exception("Longueurs incompatibles");
Console.WriteLine("Variance nulle et dimensions : contrôles OK");

LinearRegression - RandomForest: t=2.662, p unilatérale=0.0130, effet borné=0.774, seuil=0.0167


LinearRegression - GradientBoosting: t=2.097, p unilatérale=0.0327, effet borné=0.693, seuil=0.0167


RandomForest - GradientBoosting: t=-0.036, p unilatérale=0.5140, effet borné=-0.017, seuil=0.0167


Contrôle indépendant : t=2.108185, p=0.051350, p inverse=0.948650, effet=0.816497


Variance nulle et dimensions : contrôles OK


L'effet affiché est $\bar d/\sqrt{\bar d^2+s_d^2}$, une transformation bornée du d apparié, et non une corrélation rank-biserial. Le seuil ajuste les trois comparaisons ; il ne démontre pas leur importance pratique. Pour régression linéaire contre forêt, p = 0,0130 passe sous le seuil 0,0167 : c'est la seule des trois paires que la correction retient. Régression linéaire contre boosting (p = 0,0327) passerait un seuil nominal de 0,05, mais pas le seuil corrigé. Pour forêt contre boosting, p = 0,5140 dépasse le seuil 0,0167. Les valeurs observées dépendent des familles et paramètres utilisés.

### Exercice 3 — Stabilité Monte-Carlo

Pour les budgets 100, 500, 1000 et 5000, appeler dix fois le fournisseur de bornes avec des graines 0 à 9, puis retourner les écarts-types de chaque borne. Augmenter B réduit l'erreur Monte-Carlo des quantiles sous des hypothèses de régularité, pas nécessairement la largeur de l'intervalle.

In [10]:
Dictionary<int,double[]> StabilityExercise(int[] budgets,int repeats,Func<int,int,(double lo,double hi)> provider) {
    // TODO : ecarts-types des bornes par budget avec les graines 0..repeats-1.
    return null;
}
int[] budgets={100,500,1000,5000};
foreach(double scale in new[]{1.0,3.0}) {
 (double lo,double hi) Fixture(int budget,int seed)=>(scale*seed/(double)budget,2+scale*2.0*seed/budget);
 var answer=StabilityExercise(budgets,10,Fixture);
 if(answer==null) Console.WriteLine("Exercice 3 à compléter");
 else {
   bool ok=answer.Count==budgets.Length && budgets.All(b=>answer.ContainsKey(b) && answer[b].Length==2 && answer[b].All(double.IsFinite)
     && Math.Abs(answer[b][0]-scale*Math.Sqrt(8.25)/b)<1e-12 && Math.Abs(answer[b][1]-2*scale*Math.Sqrt(8.25)/b)<1e-12);
   Console.WriteLine($"Contrôle : {ok}");
 }
}

Exercice 3 à compléter


Exercice 3 à compléter


## À retenir

Un classement ou un score isolé ne prouve pas à lui seul une supériorité ; les dix plis et le bootstrap répondent à des questions distinctes. Le bootstrap de test conditionne sur les modèles entraînés. La correction Nadeau–Bengio est approximative, et Bonferroni contrôle le seuil de trois tests.

## Vérifiez votre compréhension

1. Pourquoi les scores des plis peuvent-ils être dépendants ?
2. Quelle variabilité le bootstrap du test laisse-t-il de côté ?
3. Pourquoi la largeur de l'IC ne doit-elle pas tendre vers zéro quand B augmente ?

<details><summary>Réponses</summary>

1. Les ensembles d'entraînement se recouvrent. 2. La variabilité du réentraînement. 3. Elle représente aussi une incertitude statistique, distincte de l'erreur Monte-Carlo.

</details>

## Pour aller plus loin

- Nadeau et Bengio (2003), *Inference for the Generalization Error*.
- Efron (1979), *Bootstrap methods*.
- [ML-04 — Évaluation](ML-04-Evaluation-CSharp.ipynb).